In [1]:
import pandas as pd

In [ ]:
pip install pdfplumber

#Solar
```
 Conexão e acesso	REN 1.000 + a CP 033/2026, aberta agora
 Geração distribuída
```
#Eólica
	Cortes de geração	a norma de 2026 mais relevante (Portaria MME 140)
	Eólica offshore	lei nova + regulamentação de 2026

#Armazenamento	Autorização	REN 1.161/2026
	Conexão e faturamento

#1 ) WEB SCRAPER que irá pegar os dados diariamente do diário oficial da união:

In [ ]:
"""
Energy Start — Coletor diário do DOU (sem ZIP, sem INLABS)

Usa a página pública de leitura do jornal da Imprensa Nacional:
    https://www.in.gov.br/leiturajornal?data=DD-MM-AAAA&secao=do1
Ela traz, dentro do HTML, uma lista em JSON com todas as publicações do dia.
Cada publicação completa fica em:
    https://www.in.gov.br/web/dou/-/<urlTitle>

ATENÇÃO: não é uma API documentada. Se o site mudar, rode inspecionar_dia()
e ajuste os nomes dos campos.

Instalar:  pip install requests pandas
Uso no Colab:
    df = coletar_dia("21-09-2026")                    # um dia
    df = coletar_periodo("01-09-2026", "21-09-2026")  # vários dias
    df = classificar_df(df)                           # e segue para o classificador
"""

import html
import json
import re
import time
from datetime import datetime, timedelta

import pandas as pd
import requests

BASE = "https://www.in.gov.br"
CABECALHO = {"User-Agent": "Mozilla/5.0 (EnergyStart - hackathon COPPE)"}

# Quem entra e quem sai (aprendido no CSV de agosto: ANP e ANM passavam pelo filtro do MME)
ORGAOS_INCLUIR = ["Agência Nacional de Energia Elétrica", "Ministério de Minas e Energia",
                  "Conselho Nacional de Política Energética"]
ORGAOS_EXCLUIR = ["Agência Nacional do Petróleo", "Agência Nacional de Mineração"]


# ---------------------------------------------------------------------------
# 1. Lista de publicações do dia
# ---------------------------------------------------------------------------

def _json_da_pagina(pagina_html: str) -> list:
    m = re.search(r'<script[^>]*id="params"[^>]*>(.*?)</script>', pagina_html, flags=re.S)
    if not m:
        return []
    return json.loads(m.group(1)).get("jsonArray", [])


def listar_dia(data: str, secao: str = "do1") -> list:
    """data no formato DD-MM-AAAA. Devolve a lista bruta de publicações."""
    r = requests.get(f"{BASE}/leiturajornal", params={"data": data, "secao": secao},
                     headers=CABECALHO, timeout=60)
    r.raise_for_status()
    return _json_da_pagina(r.text)


def _interessa(item: dict) -> bool:
    orgao = item.get("hierarchyStr", "") or ""
    return (any(o in orgao for o in ORGAOS_INCLUIR)
            and not any(o in orgao for o in ORGAOS_EXCLUIR))


# ---------------------------------------------------------------------------
# 2. Texto completo de cada publicação
# ---------------------------------------------------------------------------

def _limpar(t: str) -> str:
    t = re.sub(r"<(script|style)[^>]*>.*?</\1>", " ", t, flags=re.S | re.I)
    t = re.sub(r"</p>|<br\s*/?>", "\n", t, flags=re.I)
    t = re.sub(r"<[^>]+>", " ", t)
    t = html.unescape(t)
    t = re.sub(r"[ \t\xa0]+", " ", t)
    return re.sub(r"\s*\n\s*", "\n", t).strip()


def baixar_texto(url_title: str) -> str:
    r = requests.get(f"{BASE}/web/dou/-/{url_title}", headers=CABECALHO, timeout=60)
    r.raise_for_status()
    # o corpo da matéria fica na div "texto-dou"; se não achar, usa a página inteira
    m = re.search(r'<div[^>]*class="[^"]*texto-dou[^"]*"[^>]*>(.*?)</div>\s*</div>',
                  r.text, flags=re.S)
    return _limpar(m.group(1) if m else r.text)


# ---------------------------------------------------------------------------
# 3. Coleta de um dia ou de um período
# ---------------------------------------------------------------------------

def coletar_dia(data: str, pausa: float = 1.5) -> pd.DataFrame:
    itens = [i for i in listar_dia(data) if _interessa(i)]
    linhas = []
    for item in itens:
        try:
            texto = baixar_texto(item["urlTitle"])
        except Exception as e:
            print(f"  [erro ao baixar] {item.get('title')}: {e}")
            continue
        linhas.append({
            "data": item.get("pubDate", data),
            "orgao": item.get("hierarchyStr", ""),
            "tipo": item.get("artType", ""),
            "titulo": item.get("title", "") or item.get("titulo", ""),
            "texto": texto,
            "link": f"{BASE}/web/dou/-/{item['urlTitle']}",
        })
        time.sleep(pausa)  # educação com o servidor do governo
    print(f"{data}: {len(itens)} publicações de ANEEL/MME/CNPE")
    return pd.DataFrame(linhas)


def coletar_periodo(inicio: str, fim: str) -> pd.DataFrame:
    d = datetime.strptime(inicio, "%d-%m-%Y")
    ultimo = datetime.strptime(fim, "%d-%m-%Y")
    partes = []
    while d <= ultimo:
        if d.weekday() < 5:  # o DOU ordinário sai de segunda a sexta
            try:
                partes.append(coletar_dia(d.strftime("%d-%m-%Y")))
            except Exception as e:
                print(f"{d:%d-%m-%Y}: falhou ({e})")
        d += timedelta(days=1)
    partes = [p for p in partes if len(p)]
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()


# ---------------------------------------------------------------------------
# 4. Diagnóstico, se algo parar de funcionar
# ---------------------------------------------------------------------------

def inspecionar_dia(data: str):
    itens = listar_dia(data)
    print(f"{len(itens)} publicações no dia. Campos do primeiro item:")
    if itens:
        print(json.dumps(itens[0], ensure_ascii=False, indent=2)[:1500])

#1.1) Teste com Dados Diretos do dou:

In [123]:
df = pd.read_csv("dou_agosto.csv")

In [124]:
df.head()

,data,orgao,tipo,titulo,texto,link
0,03/08/2026,Ministério de Minas e Energia/Agência Nacional...,Despacho,DESPACHO,DESPACHO\nFase de Requerimento de Pesquisa\nIn...,http://pesquisa.in.gov.br/imprensa/jsp/visuali...
1,03/08/2026,Ministério de Minas e Energia/Agência Nacional...,Despacho,"DESPACHO SDL-ANP Nº 1.160, DE 31 DE JULHO DE 2026","DESPACHO SDL-ANP Nº 1.160, DE 31 DE JULHO DE 2...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...
2,03/08/2026,Ministério de Minas e Energia/Agência Nacional...,Despacho,"DESPACHO SDL-ANP Nº 1.161, DE 31 DE JULHO DE 2026","DESPACHO SDL-ANP Nº 1.161, DE 31 DE JULHO DE 2...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...
3,03/08/2026,Ministério de Minas e Energia/Agência Nacional...,Despacho,"DESPACHO SDL-ANP Nº 1.162, DE 31 DE JULHO DE 2026","DESPACHO SDL-ANP Nº 1.162, DE 31 DE JULHO DE 2...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...
4,03/08/2026,Ministério de Minas e Energia/Secretaria Nacio...,Portaria,"PORTARIA SNTEP/MME Nº 3.168, DE 30 DE JULHO DE...","PORTARIA SNTEP/MME Nº 3.168, DE 30 DE JULHO DE...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...


In [125]:
df.iloc[-1]

,1569
data,31/08/2026
orgao,Ministério de Minas e Energia/Agência Nacional...
tipo,Despacho
titulo,Despacho
texto,Despacho\nRelação nº 190/2026\nFase de Autoriz...
link,http://pesquisa.in.gov.br/imprensa/jsp/visuali...


In [127]:
df["titulo"] = df["titulo"].fillna("")
df["primeira_palavra"] = df["titulo"].str.split().str[0].str.upper()
df["primeira_palavra"].value_counts()

,count
primeira_palavra,
DESPACHO,1207
ALVARÁ,216
AUTORIZAÇÃO,51
RESOLUÇÃO,49
RETIFICAÇÃO,29
PORTARIA,13
SÚMULA,1
RETIFICAÇÕES,1


In [128]:
df_reso = df[df["primeira_palavra"]=="RESOLUÇÃO"]
df_reso.reset_index(drop=True, inplace=True)
df_reso

,data,orgao,tipo,titulo,texto,link,primeira_palavra
0,03/08/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,"RESOLUÇÃO AUTORIZATIVA Nº 16.721, DE 28 DE JUL...","RESOLUÇÃO AUTORIZATIVA Nº 16.721, DE 28 DE JUL...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...,RESOLUÇÃO
1,03/08/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,"RESOLUÇÃO AUTORIZATIVA Nº 16.723, DE 28 DE JUL...","RESOLUÇÃO AUTORIZATIVA Nº 16.723, DE 28 DE JUL...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...,RESOLUÇÃO
2,03/08/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,"RESOLUÇÃO AUTORIZATIVA Nº 16.724, DE 28 DE JUL...","RESOLUÇÃO AUTORIZATIVA Nº 16.724, DE 28 DE JUL...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...,RESOLUÇÃO
3,03/08/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,"RESOLUÇÃO AUTORIZATIVA Nº 16.725, DE 28 DE JUL...","RESOLUÇÃO AUTORIZATIVA Nº 16.725, DE 28 DE JUL...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...,RESOLUÇÃO
4,03/08/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,"RESOLUÇÃO AUTORIZATIVA Nº 16.726, DE 28 DE JUL...","RESOLUÇÃO AUTORIZATIVA Nº 16.726, DE 28 DE JUL...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...,RESOLUÇÃO
5,04/08/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,"Resolução ANM Nº 245, DE 31 DE julho DE 2026","Resolução ANM Nº 245, DE 31 DE julho DE 2026\n...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...,RESOLUÇÃO
6,04/08/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,"RESOLUÇÃO AUTORIZATIVA Nº 16.728, DE 28 DE JUL...","RESOLUÇÃO AUTORIZATIVA Nº 16.728, DE 28 DE JUL...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...,RESOLUÇÃO
7,04/08/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,"RESOLUÇÃO AUTORIZATIVA Nº 16.729, DE 28 DE JUL...","RESOLUÇÃO AUTORIZATIVA Nº 16.729, DE 28 DE JUL...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...,RESOLUÇÃO
8,04/08/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,"RESOLUÇÃO AUTORIZATIVA Nº 16.730, DE 28 DE JUL...","RESOLUÇÃO AUTORIZATIVA Nº 16.730, DE 28 DE JUL...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...,RESOLUÇÃO
9,07/08/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,"Resolução ANM Nº 246, DE 6 DE agosto DE 2026","Resolução ANM Nº 246, DE 6 DE agosto DE 2026\n...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...,RESOLUÇÃO


# O que importa

1. Resolução (49): a mais importante. É onde as regras mudam. Dentro dela há três tipos, e todos servem para vocês:

Normativa (REN): cria ou altera regras gerais, como as de geração distribuída e conexão. É o coração do produto.
Homologatória (REH): reajuste de tarifa das distribuidoras, que muda o valor da compensação solar.
Autorizativa (REA): autoriza usinas específicas, o que conversa com a subárea Outorga eólica.

2. Portaria (13): média. Algumas do MME definem diretrizes importantes; outras são administrativas. São poucas, então deixem passar e o classificador decide.

3. Despacho (1.207): pouca coisa útil no meio de muito volume. A maioria trata de casos individuais: registrar uma usina, aprovar um documento, decidir um processo. Alguns interessam (registro de usina eólica ou solar, decisões sobre cortes de geração), mas a maior parte não. Aqui vale um filtro de palavras-chave antes do LLM, para não gastar classificação com mil despachos irrelevantes.

4. Retificação (30): baixa, mas não descartem. Corrige um ato anterior. Se corrigir uma resolução normativa, importa.

In [129]:
df["subtipo"] = df["titulo"].str.upper().str.extract(r"RESOLUÇÃO\s+(\w+)")
df["subtipo"].value_counts()

df_teste = df[df["subtipo"].isin(["NORMATIVA", "HOMOLOGATÓRIA"])]
len(df_teste)

6

In [130]:
for n in range(len(df_teste)):

  print(df_teste["texto"].iloc[n])

RESOLUÇÃO HOMOLOGATÓRIA Nº 3.601, DE 11 DE AGOSTO DE 2026
O DIRETOR-GERAL DA AGÊNCIA NACIONAL DE ENERGIA ELÉTRICA - ANEEL, com base no art. 16, IV, do Regimento Interno da ANEEL, resolve:
Processo nº: 48500.016427/2026-62. Interessados: Companhia de Eletricidade do Amapá - Equatorial CEA (CNPJ nº 05.965.546/0001-09), Câmara de Comercialização de Energia Elétrica - CCEE, UHE Coaracy Nunes, Linhas de Macapá Transmissora de Energia S.A. - LMTE, Energisa Amapá Transmissora de Energia S.A. - EAP, concessionárias e permissionárias de distribuição, consumidores, usuários e agentes do Setor. Objeto: Prorrogar a vigência das tarifas da Companhia de Eletricidade do Amapá - Equatorial CEA, previstas na Resolução Homologatória nº 3.572, de 7 de abril de 2026, pelo período de 13 de dezembro de 2026 a 24 de maio de 2027. A íntegra desta Resolução e de seus anexos estão juntados aos autos e disponíveis no endereço eletrônico legislacao.aneel.gov.br.
SANDOVAL DE ARAÚJO FEITOSA NETO
RESOLUÇÃO HOMOLOGAT

#Como o texto em si da norma está no site da ENEEL, vou passar para pegar os dados na Lei.org que possui todas as esferas Federal , estadual e municipal

# 2 ) Aneel + Busca do claude

In [64]:
"""
Energy Start — Pré-processamento dos PDFs (Leis.org / ANEEL)

Lê todos os PDFs de uma pasta, limpa o texto e devolve um DataFrame
no mesmo formato que o classificar_df() espera (coluna "texto").

Limpezas feitas:
  1. remove o rodapé "Leis.org/aneel ... Gerado em ..." e a numeração "1/15";
  2. remove a redação ANTIGA quando o texto consolidado traz as duas
     versões ("... (Redação dada pela Resolução Normativa nº X)");
  3. remove dispositivos revogados;
  4. separa a lista "Atos que alteram... / Atos que são alterados..."
     do final, que é do site e não da norma.

Instalar:  pip install pdfplumber pandas
Uso:
    df = carregar_pdfs(r"C:\\caminho\\da\\pasta")
    novidades = df[df["uso"] == "novidade"]      # vão para o classificador
    estoque   = df[df["uso"] == "estoque"]       # vão para a base do copiloto
"""

import re
from pathlib import Path

import pandas as pd
import pdfplumber

LIMITE_ESTOQUE = 150_000  # acima disso (ex.: REN 1.000), é norma-base, não novidade

# Início de um dispositivo: "Art. 12.", "Art. 655-A", "§ 2º", "Parágrafo único",
# "XXIX-A -", "IV -", "a)"
ROTULO = re.compile(
    r"^\s*(Art\.\s*\d+[º°]?(?:-[A-Z]+)?|§\s*\d+[º°]?|Parágrafo único|"
    r"[IVXLC]+(?:-[A-Z]+)?\s+-|[a-z]\))"
)


# ---------------------------------------------------------------------------
# Leitura
# ---------------------------------------------------------------------------

def ler_pdf(caminho: Path) -> tuple[str, int]:
    """pdfplumber mantém o 'Art. 1º' no lugar certo (o pypdf solta o número do texto)."""
    with pdfplumber.open(caminho) as pdf:
        paginas = [p.extract_text() or "" for p in pdf.pages]
    return "\n".join(paginas), len(paginas)


# ---------------------------------------------------------------------------
# Limpeza
# ---------------------------------------------------------------------------

def remover_rodapes(texto: str) -> str:
    texto = re.sub(r"Leis\.org/aneel\s*-.*?Gerado em:\s*[\d/]+\s*[\d:]+", "", texto)
    texto = re.sub(r"^\s*\d+/\d+\s*$", "", texto, flags=re.M)
    return texto


def separar_atos_relacionados(texto: str) -> tuple[str, str]:
    m = re.search(r"Atos que alteram, regulamentam ou revogam", texto)
    if not m:
        return texto, ""
    return texto[: m.start()].strip(), texto[m.start():].strip()


def _chave(rotulo: str) -> str:
    """'Art. 176' e 'Art. 176.' viram a mesma chave."""
    return re.sub(r"[\s.º°]", "", rotulo).lower()


def remover_redacoes_antigas(texto: str) -> str:
    # 1) junta as linhas quebradas em dispositivos inteiros
    blocos: list[str] = []
    for linha in texto.splitlines():
        if not linha.strip():
            continue
        if ROTULO.match(linha) or not blocos:
            blocos.append(linha.strip())
        else:
            blocos[-1] += " " + linha.strip()

    # 2) quando aparece a redação nova, apaga a antiga com o mesmo rótulo
    manter = [True] * len(blocos)
    for i, bloco in enumerate(blocos):
        novo = "(Redação dada pela" in bloco
        revogado = "(Revogado pel" in bloco or "(Revogada pel" in bloco
        if revogado:
            manter[i] = False
        if not (novo or revogado):
            continue
        m = ROTULO.match(bloco)
        if not m:
            continue
        chave = _chave(m.group(1))
        for j in range(i - 1, max(i - 15, -1), -1):  # procura a versão antiga logo antes
            mj = ROTULO.match(blocos[j])
            if mj and _chave(mj.group(1)) == chave and manter[j]:
                manter[j] = False
                break

    return "\n".join(b for b, ok in zip(blocos, manter) if ok)


def extrair_titulo(texto: str) -> str:
    m = re.search(r"RESOLUÇÃO NORMATIVA[^\n]*?Nº\s*[\d.]+,?\s*DE\s+[^\n(]+?\d{4}", texto, flags=re.I)
    return m.group(0).strip() if m else ""


def extrair_data_dou(texto: str) -> str:
    """Leis.org informa: 'Este texto não substitui o publicado no D.O. de 08.04.2025'."""
    m = re.search(r"publicado no D\.?O\.?\s*de\s*(\d{2}\.\d{2}\.\d{4})", texto)
    return m.group(1).replace(".", "/") if m else ""


# ---------------------------------------------------------------------------
# Função principal
# ---------------------------------------------------------------------------

def carregar_pdfs(pasta: str) -> pd.DataFrame:
    linhas = []
    for caminho in sorted(Path(pasta).glob("*.pdf")):
        bruto, n_paginas = ler_pdf(caminho)
        if not bruto.strip():
            print(f"[sem texto] {caminho.name} — PDF escaneado?")
            continue

        texto = remover_rodapes(bruto)
        titulo = extrair_titulo(texto)  # antes de juntar as linhas, senão pega a ementa junto
        if not titulo:
            print(f"[ignorado] {caminho.name} — não parece uma norma")
            continue
        texto, atos = separar_atos_relacionados(texto)
        texto = remover_redacoes_antigas(texto)

        linhas.append({
            "arquivo": caminho.name,
            "titulo": titulo,
            "data": extrair_data_dou(texto),
            "paginas": n_paginas,
            "caracteres": len(texto),
            "uso": "estoque" if len(texto) > LIMITE_ESTOQUE else "novidade",
            "texto": texto,
            "atos_relacionados": atos,
        })

    df = pd.DataFrame(linhas)
    print(df[["arquivo", "paginas", "caracteres", "uso"]].to_string(index=False))
    return df


if __name__ == "__main__":
    import sys
    pasta = "Normas"
    carregar_pdfs(pasta).to_csv("normas_pdf.csv", index=False, encoding="utf-8-sig")

[ignorado] lei-14300-6-janeiro-2022-792217-normaatualizada-pl.pdf — não parece uma norma
                                   arquivo  paginas  caracteres      uso
Resolucao-normativa-1115-2025-Aneel-BR.pdf        3        4463 novidade
Resolucao-normativa-1147-2025-Aneel-BR.pdf       15       25042 novidade
Resolucao-normativa-1148-2026-Aneel-BR.pdf       15       21851 novidade
Resolucao-normativa-1150-2026-Aneel-BR.pdf        4        5263 novidade
       aren2021956_prodist_modulo_3_v8.pdf       41        1528 novidade


In [65]:
df_normas=pd.read_csv("normas_pdf.csv")
df_normas


,arquivo,titulo,data,paginas,caracteres,uso,texto,atos_relacionados
0,Resolucao-normativa-1115-2025-Aneel-BR.pdf,"RESOLUÇÃO NORMATIVA ANEEL Nº 1.115, DE 1º DE A...",08/04/2025,3,4463,novidade,AGÊNCIA NACIONAL DE ENERGIA ELÉTRICA - ANEEL R...,"Atos que alteram, regulamentam ou revogam esta..."
1,Resolucao-normativa-1147-2025-Aneel-BR.pdf,"RESOLUÇÃO NORMATIVA Nº 1.147, DE 9 DE DEZEMBRO...",16/12/2025,15,25042,novidade,AGÊNCIA NACIONAL DE ENERGIA ELÉTRICA - ANEEL R...,"Atos que alteram, regulamentam ou revogam esta..."
2,Resolucao-normativa-1148-2026-Aneel-BR.pdf,"RESOLUÇÃO NORMATIVA ANEEL Nº 1.148, DE 27 DE J...",20/02/2026,15,21851,novidade,AGÊNCIA NACIONAL DE ENERGIA ELÉTRICA - ANEEL R...,"Atos que alteram, regulamentam ou revogam esta..."
3,Resolucao-normativa-1150-2026-Aneel-BR.pdf,"RESOLUÇÃO NORMATIVA ANEEL Nº 1.150, DE 27 DE J...",02/02/2026,4,5263,novidade,AGÊNCIA NACIONAL DE ENERGIA ELÉTRICA - ANEEL R...,"Atos que alteram, regulamentam ou revogam esta..."
4,aren2021956_prodist_modulo_3_v8.pdf,"RESOLUÇÃO NORMATIVA ANEEL Nº 956, DE 7 DE DEZE...",NaN,41,1528,novidade,"ANEXO III DA RESOLUÇÃO NORMATIVA ANEEL Nº 956,...",NaN


In [11]:
df_normas["texto"][0]

'AGÊNCIA NACIONAL DE ENERGIA ELÉTRICA - ANEEL RESOLUÇÃO NORMATIVA ANEEL Nº 1.000, DE 7 DE DEZEMBRO DE 2021(*) (Vide Despacho nº 1.731/2025 - Aplicação da Tarifa Social de Energia Elétrica, alterada pela Medida Provisória nº1 .300/2025, nos termos dos arts. 179, §§ 1º e 2º, e art. 291, inciso III) Despacho 2.006/2024: Decisão Judicial - suspensos os efeitos referentes ao prazo de 60 (sessenta) ciclos estabelecidos no inciso II do art. 323. Estabelece as Regras de Prestação do Serviço Público de Distribuição de Energia Elétrica; revoga as Resoluções Normativas ANEEL nº 414, de 9 de setembro de 2010; nº 470, de 13 de dezembro de 2011; nº 901, de 8 de dezembro de 2020 e dá outras providências. O DIRETOR-GERAL DA AGÊNCIA NACIONAL DE ENERGIA ELÉTRICA - ANEEL, no uso de suas atribuições regimentais, de acordo com a deliberação da Diretoria, tendo em vista o disposto na Lei nº 9.427, de 26 de dezembro de 1996, no Decreto nº 2.335, de 6 de outubro de 1997 e o que consta do Processo nº 48500.005

# 3- Modelo

In [67]:
# ============================================================
# Energy Start — Classificador (1º LLM) usando a API da NVIDIA
# Cole esta célula inteira no Colab e rode.
# A chave fica nos Secrets do Colab (ícone 🔑) com o nome NVIDIA_KEY.
# ============================================================

import json
import re
import time

import pandas as pd
import requests
from google.colab import userdata

URL = "https://integrate.api.nvidia.com/v1/chat/completions"
MODELO = "google/gemma-4-31b-it"
CHAVE = "Bearer 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"

# ------------------------------------------------------------
# 1. Taxonomia: áreas e subáreas do protótipo
# ------------------------------------------------------------
TAXONOMIA = {
    "Solar": {
        "Geração distribuída": "micro e minigeração distribuída, sistema de compensação de energia, "
        "créditos, Fio B, geração compartilhada, autoconsumo remoto, Lei 14.300, reajustes "
        "tarifários que mudam o valor da compensação",
        "Conexão e acesso": "pedido de conexão à rede, prazos da distribuidora, recusa por "
        "inversão de fluxo, requisitos técnicos de conexão",
    },
    "Eólica": {
        "Cortes de geração": "redução de geração determinada pelo operador (curtailment), "
        "regras de ressarcimento",
        "Tarifas e encargos": "tarifas de uso da transmissão, encargos pagos por geradores",
    },
    "Armazenamento": {
        "Autorização de armazenamento": "requisitos e procedimentos para autorizar sistemas de "
        "armazenamento de energia (baterias) autônomos",
        "Conexão e faturamento de armazenamento": "conexão, contratação de uso da rede, "
        "faturamento e baterias instaladas junto a usinas (colocalizadas)",
    },
}

# ------------------------------------------------------------
# 2. Prompt
# ------------------------------------------------------------
def montar_prompt():
    linhas = []
    for area, subs in TAXONOMIA.items():
        linhas.append(f"Área: {area}")
        for sub, desc in subs.items():
            linhas.append(f"  - Subárea: {sub} — {desc}")
    lista = "\n".join(linhas)

    return f"""Você é um analista regulatório do setor elétrico brasileiro.
Sua tarefa é classificar uma norma (resolução, portaria ou despacho) nas áreas e subáreas abaixo.

ÁREAS E SUBÁREAS PERMITIDAS (use somente estas, com os nomes exatamente iguais):
{lista}

REGRAS:
1. Leia o texto inteiro. Uma norma sobre outro assunto pode alterar um artigo que pertence
   a uma subárea (ex.: uma norma sobre tarifa social que muda uma regra de compensação de
   geração distribuída). Nesse caso, classifique na subárea e diga isso na justificativa.
2. Uma norma pode ter mais de uma classificação, ou nenhuma.
3. Use apenas o que está escrito no texto. Não deduza nem use conhecimento próprio.
4. Mencionar uma palavra não basta: a norma precisa criar, alterar ou revogar uma regra
   daquela subárea. Ex.: citar "recursos energéticos distribuídos" de passagem NÃO é
   geração distribuída.
5. Se nada se encaixar, devolva a lista vazia.

FORMATO DA RESPOSTA: apenas um objeto JSON, sem texto antes ou depois, sem ```:
{{"classificacoes": [{{"area": "Solar", "subarea": "Geração distribuída"}}],
  "relevancia": 0,
  "justificativa": "uma frase curta explicando a escolha"}}

"relevancia" vai de 0 a 3:
0 = não afeta nenhuma subárea
1 = afeta pouco ou indiretamente
2 = altera regra de uma subárea
3 = muda regra central de uma subárea (prazo, limite, valor, obrigação)"""

# ------------------------------------------------------------
# 3. Chamada à NVIDIA para um texto
# ------------------------------------------------------------
def _ler_json(resposta):
    limpo = re.sub(r"```(?:json)?", "", resposta)
    return json.loads(limpo[limpo.find("{"): limpo.rfind("}") + 1])


def _validar(dados):
    """Descarta qualquer área/subárea que não esteja na TAXONOMIA."""
    return [(c.get("area"), c.get("subarea")) for c in dados.get("classificacoes", [])
            if c.get("area") in TAXONOMIA and c.get("subarea") in TAXONOMIA[c.get("area")]]


def chamar_nvidia(mensagens):
    headers = {"Authorization": f"{CHAVE}", "Accept": "application/json"}
    payload = {
        "model": MODELO,
        "messages": mensagens,
        "temperature": 0,      # mesma norma -> mesma resposta
        "max_tokens": 1024,    # suficiente para o JSON
        "stream": False,
    }
    r = requests.post(URL, headers=headers, json=payload, timeout=120)
    if r.status_code in (401, 403):
        raise RuntimeError("Chave inválida ou sem permissão. Confira o Secret NVIDIA_KEY.")
    r.raise_for_status()   # outros erros (limite, servidor) sobem para nova tentativa
    return r.json()["choices"][0]["message"]["content"] or ""


def classificar_texto(texto, max_chars=40000):
    mensagens = [
        {"role": "system", "content": montar_prompt()},
        {"role": "user", "content": "NORMA A CLASSIFICAR:\n\n" + texto[:max_chars]},
    ]
    for _ in range(2):  # nova tentativa se o JSON vier quebrado
        try:
            dados = _ler_json(chamar_nvidia(mensagens))
            pares = _validar(dados)
            return {
                "area": sorted({a for a, _ in pares}),
                "subarea": [s for _, s in pares],
                "relevancia": dados.get("relevancia", 0) if pares else 0,
                "justificativa": dados.get("justificativa", ""),
            }
        except (json.JSONDecodeError, ValueError):
            continue
    return {"area": [], "subarea": [], "relevancia": 0, "justificativa": "ERRO: resposta inválida"}

# ------------------------------------------------------------
# 4. Classificar o DataFrame inteiro
# ------------------------------------------------------------
def classificar_df(df, coluna_texto="texto", pausa=7):
    resultados = []
    for i, texto in enumerate(df[coluna_texto].astype(str), start=1):
        for tentativa in range(3):
            try:
                r = classificar_texto(texto)
                break
            except RuntimeError:
                raise                      # chave errada: para na hora
            except Exception as e:
                print(f"  erro na tentativa {tentativa + 1}: {e} — esperando 30s")
                time.sleep(30)
        else:
            r = {"area": [], "subarea": [], "relevancia": 0,
                 "justificativa": "ERRO: limite ou falha da API"}
        print(f"[{i}/{len(df)}] {r['subarea'] or 'nenhuma'} (relevância {r['relevancia']})")
        resultados.append(r)
        time.sleep(pausa)
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(resultados)], axis=1)

# ------------------------------------------------------------
# Uso:
#   teste = classificar_df(novidades.head(1))
#   teste[["titulo", "area", "subarea", "relevancia", "justificativa"]]
# ------------------------------------------------------------

In [68]:
teste = classificar_df(df_normas)
#   teste[["titulo", "area", "subarea", "relevancia", "justificativa"]]

[1/5] nenhuma (relevância 0)
[2/5] nenhuma (relevância 0)
[3/5] nenhuma (relevância 0)
[4/5] nenhuma (relevância 0)
[5/5] ['Conexão e acesso'] (relevância 3)


In [89]:
relevantes = teste[teste["relevancia"] > 0]
relevantes


,arquivo,titulo,data,paginas,caracteres,uso,texto,atos_relacionados,area,subarea,relevancia,justificativa
4,aren2021956_prodist_modulo_3_v8.pdf,"RESOLUÇÃO NORMATIVA ANEEL Nº 956, DE 7 DE DEZE...",NaN,41,1528,novidade,"ANEXO III DA RESOLUÇÃO NORMATIVA ANEEL Nº 956,...",NaN,[Solar],[Conexão e acesso],3,O texto estabelece a estrutura de requisitos t...


In [95]:
relevantes["texto"][4]

'ANEXO III DA RESOLUÇÃO NORMATIVA ANEEL Nº 956, DE 7 DE DEZEMBRO DE 2021 PROCEDIMENTOS DE DISTRIBUIÇÃO DE ENERGIA ELÉTRICA NO SISTEMA ELÉTRICO NACIONAL – PRODIST MÓDULO 3 - CONEXÃO AO SISTEMA DE DISTRIBUIÇÃO DE ENERGIA ELÉTRICA Seção 3.0 Introdução Conteúdo 1. Além desta seção introdutória, este módulo está estruturado da seguinte forma: Seção 3.1 – Requisitos para Conexão de Microgeração e Minigeração Distribuída: estabelece requisitos técnicos para conexão de microgeração e minigeração distribuída ao sistema de distribuição; Seção 3.2 – Requisitos para Conexão de Central Geradora: estabelece requisitos técnicos para conexão de central geradora ao sistema de distribuição; Seção 3.3 – Requisitos de Projeto das Instalações de Conexão: define os requisitos a serem observados para elaboração de projetos de instalações de conexão; Seção 3.4 – Requisitos dos Sistemas de Proteção para demais Usuários: define requisitos gerais de proteção para usuários que não se enquadram nas Seções 3.1 e 3.

In [ ]:
# Vou precisar guardar no banco, o texto, a data, a área e a subárea Para jogar nos alertas.

# 4 ) Testando o Scraping junto com o classificador de relevância:


In [97]:
df = coletar_dia("21-09-2026")

21-09-2026: 75 publicações de ANEEL/MME/CNPE
[1/75] nenhuma (relevância 0)
[2/75] nenhuma (relevância 0)
[3/75] nenhuma (relevância 0)
[4/75] nenhuma (relevância 0)
[5/75] nenhuma (relevância 0)
[6/75] nenhuma (relevância 0)


KeyboardInterrupt: 

In [102]:
df_class = classificar_df(df)

# Resolção Autorizativa só passa se tiver o tema citado, caso contrario nao vale a pena
normativos = df["titulo"].str.contains("NORMATIVA|HOMOLOGATÓRIA", case=False, na=False)
temas = "geração distribuída|microgeração|minigeração|compensação de energia|conexão|constrained|corte de geração|eólic|offshore|armazenamento|bateria"
tem_tema = df["texto"].str.contains(temas, case=False, na=False)
para_classificar = df[normativos | tem_tema]

[1/75] nenhuma (relevância 0)
[2/75] nenhuma (relevância 0)
[3/75] nenhuma (relevância 0)
[4/75] nenhuma (relevância 0)
[5/75] nenhuma (relevância 0)


KeyboardInterrupt: 

In [104]:
df["texto"][30]

'DESPACHO Nº 3.622, DE 15 DE SETEMBRO DE 2026\nO DIRETOR-GERAL DA AGÊNCIA NACIONAL DE ENERGIA ELÉTRICA - ANEEL, no uso de suas atribuições regimentais, tendo em vista deliberação da Diretoria e o que consta do Processo nº 48500.003422/2026-70, decide:\nconhecer e, no mérito, negar provimento ao Pedido de Impugnação apresentado pela Barra Bonita Óleo e Gás S.A. - Barra Bonita I, cadastrada sob o CNPJ 22.881.417/0001-43, contra a decisão do Conselho de Administração da Câmara de Comercialização de Energia Elétrica - CAd/CCEE, em sua 1.500ª reunião, referente à penalidade por insuficiência de lastro de energia de reserva.\nSANDOVAL DE ARAÚJO FEITOSA NETO\nEste conteúdo não substitui o publicado na versão certificada.\nBorda do rodapé\nLogo da Imprensa'

In [115]:
dados = df[~df["tipo"].isin(["Despacho","Portaria"])]
dados.reset_index(drop=True,inplace=True)

In [121]:
dados

,data,orgao,tipo,titulo,texto,link
0,21/09/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,"RESOLUÇÃO AUTORIZATIVA Nº 16.786, DE 15 DE SET...","RESOLUÇÃO AUTORIZATIVA Nº 16.786, DE 15 DE SET...",https://www.in.gov.br/web/dou/-/resolucao-auto...
1,21/09/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,"RESOLUÇÃO AUTORIZATIVA Nº 16.780, DE 15 DE SET...","RESOLUÇÃO AUTORIZATIVA Nº 16.780, DE 15 DE SET...",https://www.in.gov.br/web/dou/-/resolucao-auto...
2,21/09/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,"RESOLUÇÃO AUTORIZATIVA Nº 16.777, DE 15 DE SET...","RESOLUÇÃO AUTORIZATIVA Nº 16.777, DE 15 DE SET...",https://www.in.gov.br/web/dou/-/resolucao-auto...
3,21/09/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,"RESOLUÇÃO AUTORIZATIVA Nº 16.784, DE 15 DE SET...","RESOLUÇÃO AUTORIZATIVA Nº 16.784, DE 15 DE SET...",https://www.in.gov.br/web/dou/-/resolucao-auto...
4,21/09/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,"RESOLUÇÃO AUTORIZATIVA Nº 16.776, DE 15 DE SET...","RESOLUÇÃO AUTORIZATIVA Nº 16.776, DE 15 DE SET...",https://www.in.gov.br/web/dou/-/resolucao-auto...
5,21/09/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,"RESOLUÇÃO AUTORIZATIVA Nº 16.779, DE 15 DE SET...","RESOLUÇÃO AUTORIZATIVA Nº 16.779, DE 15 DE SET...",https://www.in.gov.br/web/dou/-/resolucao-auto...
6,21/09/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,"RESOLUÇÃO AUTORIZATIVA Nº 16.785, DE 15 DE SET...","RESOLUÇÃO AUTORIZATIVA Nº 16.785, DE 15 DE SET...",https://www.in.gov.br/web/dou/-/resolucao-auto...
7,21/09/2026,Ministério de Minas e Energia/Agência Nacional...,Resolução,RESOLUÇÕES AUTORIZATIVAS DE 15 DE SETEMBRO DE ...,RESOLUÇÕES AUTORIZATIVAS DE 15 DE SETEMBRO DE ...,https://www.in.gov.br/web/dou/-/resolucoes-aut...


In [1]:
dados["texto"]

NameError: name 'dados' is not defined